In [0]:

%python
# Widgets del proyecto
dbutils.widgets.removeAll()
dbutils.widgets.text("storageName", "sanvarkim")

In [0]:

-- Crear external location para bronze
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-bronze`
URL 'abfss://bronze@sanvarkim.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL credentialsanvar)
COMMENT 'Ubicación externa para tablas bronce';

-- Crear external location para silver
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-silver`
URL 'abfss://silver@sanvarkim.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL credentialsanvar)
COMMENT 'Ubicación externa para tablas silver';

-- Crear external location para golden
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-golden`
URL 'abfss://golden@sanvarkim.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL credentialsanvar)
COMMENT 'Ubicación externa para tablas golden';

In [0]:

-- Eliminar todos los schemas y tablas
DROP CATALOG IF EXISTS sanvar_kim CASCADE;

-- Crear catalogo bronze - silver - golden
CREATE CATALOG IF NOT EXISTS sanvar_kim
MANAGED LOCATION 'abfss://bronze@sanvarkim.dfs.core.windows.net/'
COMMENT 'Catalogo para almacenar las tablas bronze;';

CREATE CATALOG IF NOT EXISTS sanvar_kim
MANAGED LOCATION 'abfss://silver@sanvarkim.dfs.core.windows.net/'
COMMENT 'Catalogo para almacenar las tablas silver;';

CREATE CATALOG IF NOT EXISTS sanvar_kim
MANAGED LOCATION 'abfss://golden@sanvarkim.dfs.core.windows.net/'
COMMENT 'Catalogo para almacenar las tablas golden;'

In [0]:

-- Eliminar todos los schemas existentes
DROP SCHEMA IF EXISTS sanvar_kim.raw CASCADE;
DROP SCHEMA IF EXISTS sanvar_kim.bronze CASCADE;
DROP SCHEMA IF EXISTS sanvar_kim.silver CASCADE;
DROP SCHEMA IF EXISTS sanvar_kim.golden CASCADE;

In [0]:

-- Crear todos los schemas
CREATE SCHEMA IF NOT EXISTS sanvar_kim.bronze;
CREATE SCHEMA IF NOT EXISTS sanvar_kim.silver;
CREATE SCHEMA IF NOT EXISTS sanvar_kim.golden;
CREATE SCHEMA IF NOT EXISTS sanvar_kim.raw;

Tablas Silver

In [0]:

-- Crear la estructura de la tabla silver para la transformacion

CREATE TABLE IF NOT EXISTS sanvar_kim.silver.detalle_alquileres (
  Consecutivo INTEGER,
  identificacion STRING,
  nombre_inquilino  STRING,
  apellido_inquilino STRING,
  fecha_nacimiento STRING,
  sexo STRING,
  edad STRING,
  peso STRING,
  direccion STRING,
  id_apartamento STRING,
  habitaciones STRING,
  cochera STRING,
  servicios STRING,
  ubicacion STRING,
  estado  STRING,
  fecha_inicio  STRING,
  fecha_fin STRING
)

USING DELTA
LOCATION 'abfss://silver@sanvarkim.dfs.core.windows.net/'

Tablas Golden

In [0]:
-- Crear la estructura de la tabla silver para el resultado final

CREATE TABLE IF NOT EXISTS sanvar_kim.golden.load_alquileres (
  Consecutivo integer,
  identificacion STRING,
  nombre_inquilino  STRING,
  apellido_inquilino STRING,
  direccion STRING,
  id_apartamento STRING,
  habitaciones STRING,
  cochera STRING,
  servicios STRING,
  ubicacion_apt STRING
)

USING DELTA
LOCATION 'abfss://golden@sanvarkim.dfs.core.windows.net/'

Tabla Bronze

In [0]:
%sql

-- Ambiente para carga de datos
USE CATALOG sanvar_kim;
USE SCHEMA bronze;

In [0]:
%python
#Lectura del archivo de personas
df = spark.read.csv("abfss://bronze@sanvarkim.dfs.core.windows.net/persona.csv",header=True)
df.write.mode("append").option("mergeSchema", "true").saveAsTable("bronze_persona")
df_bronze_persona = spark.table("bronze_persona")
#display(df_bronze_persona)

In [0]:
%python
#Lectura del archivo de apartamento
df = spark.read.csv("abfss://bronze@sanvarkim.dfs.core.windows.net/apartamento.csv",header=True)
df.write.mode("append").option("mergeSchema", "true").saveAsTable("bronze_apartamento")
df_bronze_apartamento = spark.table("bronze_apartamento")
#display(df_bronze_apartamento)

In [0]:
%python
#Lectura del archivo de alquiler
df = spark.read.csv("abfss://bronze@sanvarkim.dfs.core.windows.net/alquiler.csv",header=True)
df.write.mode("append").option("mergeSchema", "true").saveAsTable("bronze_alquiler")
df_bronze_alquiler = spark.table("bronze_alquiler")
#display(df_bronze_alquiler)